# Dự báo Doanh thu Sản phẩm
## 3. Làm sạch Dữ liệu & Chuẩn bị Biến mục tiêu

Từ kết quả kiểm tra ở Bước 2, chúng ta tiến hành các bước làm sạch và chuẩn bị dữ liệu như sau:
1. Loại bỏ dữ liệu dở dang của tháng 3/2023 để tránh gây lệch xu hướng thời gian.
2. Tạo biến mục tiêu dự báo: `Sales_Revenue = quantity * price` (Doanh thu = Số lượng * Đơn giá).
3. Loại bỏ các biến định danh và các biến gây rò rỉ thông tin (`invoice_no`, `customer_id`, `quantity`, `price`).
4. Lưu lại bộ dữ liệu sạch để phục vụ phân tích EDA và huấn luyện mô hình.

In [ ]:
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

RAW_DATA_PATH = r'../../data/raw-data/customer_shopping_data.csv'
df = pd.read_csv(RAW_DATA_PATH)
print('Kích thước dữ liệu gốc:', df.shape)

### 3.1 Loại bỏ dữ liệu dở dang theo thời gian
Chúng ta chỉ giữ lại các giao dịch phát sinh trước ngày 01/03/2023 để đảm bảo tính nhất quán của phân tích chu kỳ tháng.

In [ ]:
df['invoice_date'] = pd.to_datetime(df['invoice_date'], format='%d/%m/%Y')
cleaned_df = df[df['invoice_date'] < '2023-03-01'].copy()
print('Kích thước sau khi lọc mốc thời gian:', cleaned_df.shape)
print('Ngày hóa đơn muộn nhất hiện tại:', cleaned_df['invoice_date'].max())

### 3.2 Khởi tạo biến mục tiêu (Sales_Revenue)
Doanh thu của mỗi hóa đơn được tính bằng tích số lượng và đơn giá sản phẩm.

In [ ]:
cleaned_df['Sales_Revenue'] = cleaned_df['quantity'] * cleaned_df['price']
print('Thống kê mô tả biến doanh thu:')
print(cleaned_df['Sales_Revenue'].describe())

### 3.3 Loại bỏ các cột không có giá trị dự báo hoặc gây rò rỉ thông tin
- `invoice_no` và `customer_id` là các mã định danh không mang ý nghĩa hành vi khách hàng.
- `quantity` và `price` bắt buộc phải loại bỏ vì biến mục tiêu `Sales_Revenue` được tính trực tiếp từ chúng. Nếu giữ lại, mô hình sẽ học được công thức nhân cơ bản này một cách hoàn hảo và dẫn đến hiện tượng **rò rỉ dữ liệu (target leakage)** phi thực tế.

In [ ]:
cleaned_df = cleaned_df.drop(columns=['invoice_no', 'customer_id', 'quantity', 'price'])
print('Các cột còn lại sau khi xử lý:', list(cleaned_df.columns))
cleaned_df.head()

### 3.4 Lưu trữ bộ dữ liệu sạch

In [ ]:
out_dir = r'../data/preprocessed-data'
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, 'preprocessed_customer_shopping_data.csv')

cleaned_df.to_csv(out_path, index=False)
print('Đã lưu dữ liệu làm sạch tại:', out_path)
print('Kích thước dữ liệu cuối cùng:', cleaned_df.shape)